<a href="https://colab.research.google.com/github/davidekim/WRAPs/blob/main/barrel_wraps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Pipeline example for creating beta barrel WRAPs**

#### This pipeline is based on the one developed and used for the WRAPs paper.

Install py3Dmol and then downgrade to python3.10 to be compatible with RFDiffusion.

In [ ]:
!pip install py3Dmol --upgrade
!sudo apt-get install python3.10-dev
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1

We need to switch to python3.10. Run the cell below, type '1' at the prompt and then press enter.

In [ ]:
!sudo update-alternatives --config python3

In [ ]:
!python --version
!ls /usr/include/python3.10/Python.h

In [ ]:
#@title setup **sushimaki** (~5-10min)
%%time
import os, time
import sys
import subprocess

def run_cmd(cmd, verbose=True):
  process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True, text=True)
  for line in iter(process.stdout.readline, ''):
    if verbose:
      sys.stdout.write(line)
      sys.stdout.flush()
  process.stdout.close()
  process.wait()

if not os.path.isdir("sushimaki"):
  print("installing sushimaki...")
  run_cmd("git clone https://github.com/davidekim/sushimaki.git")
  run_cmd("cd sushimaki; git submodule init; git submodule update --remote;")
  # install dependencies for ppi_iterative_opt submodule
  run_cmd("curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py")
  run_cmd("python get-pip.py --force-reinstall")
  run_cmd("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
  run_cmd("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  run_cmd("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  run_cmd("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  run_cmd("cd sushimaki/ppi_iterative_opt/rf_diffusion/env/SE3Transformer; pip install .")
  run_cmd("pip install biopython==1.81")
  run_cmd("pip install -U dm-haiku")
  run_cmd("pip install ml-collections")
  run_cmd('pip install --upgrade "jax[cuda12_pip]<0.6.0" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html')
  run_cmd("pip install numpy")
  run_cmd("python -m pip install --user --upgrade numpy")
  run_cmd("pip install torch jax")
  run_cmd("pip install matplotlib")
  run_cmd("pip install matplotlib-inline")
  run_cmd("pip install ipython psutil tqdm requests pandas scikit-learn dm-tree tensorflow")
  run_cmd("pip install nvidia-cublas-cu12")
  # install DeepTMHMM
  run_cmd("pip3 install --upgrade pybiolib")
  # install PyRosetta
  run_cmd("pip install pyrosetta --find-links https://west.rosettacommons.org/pyrosetta/quarterly/release")
  run_cmd("pip install py3Dmol")
print()
run_cmd("python sushimaki/sushimaki.py")

In [ ]:
#@title Download RFdiffusion checkpoint
%%time
if not os.path.exists("sushimaki/ppi_iterative_opt/rf_diffusion/models/BFF_4.pt"):
  os.system("mkdir sushimaki/ppi_iterative_opt/rf_diffusion/models")
  os.system("cd sushimaki/ppi_iterative_opt/rf_diffusion/models; wget https://files.ipd.uw.edu/pub/ppi_iterative_opt/rf_diffusion/models/BFF_4.pt")

In [ ]:
#@title Download AF2 params (~5min)
%%time
if not os.path.isdir("sushimaki/ppi_iterative_opt/af2_initial_guess/params"):
  os.system("mkdir sushimaki/ppi_iterative_opt/af2_initial_guess/params")
  os.system("cd sushimaki/ppi_iterative_opt/af2_initial_guess/params; wget https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; tar -xf alphafold_params_2022-12-06.tar")


In [ ]:
%%time
import glob
from google.colab import files

#@title Run **sushimaki** on an uploaded transmembrane target PDB (~5min)
#@markdown Clear the target_to_wrap field to upload your own PDB file.
#@markdown 2ge4A is provided as an example.

#@markdown PDB must be a single chain and have a transmembrane region.

target_to_wrap = "2ge4A" #@param {type: "string"}
target_to_wrap = target_to_wrap.split()[0]

# clear previous inputs
os.system("rm -rf input_DeepTMHMM input.pdb input_WRAP*.pdb partial_diffusion_task_file_input*.txt")
input_pdb_str = ""
if len(target_to_wrap) > 4: # must include chain id
  pdb_code = target_to_wrap[0:4]
  pdb_chain = target_to_wrap[4:]
  if not os.path.isfile(f"{pdb_code}.pdb1"):
    os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_code}.pdb1.gz")
    os.system(f"gunzip {pdb_code}.pdb1.gz")
  with open(f"{pdb_code}.pdb1") as f:
    for l in f:
      if l.startswith("ATOM") and l[20:22].strip() == pdb_chain:
        input_pdb_str += l
  with open("input.pdb", "w") as out: out.write(input_pdb_str)
else:
  uploads = files.upload()
  target_to_wrap = list(uploads.keys())[0].split('.pdb')[0].split('/')[-1].split()[0]
  input_pdb_str = uploads[list(uploads.keys())[0]]
  with open("input.pdb", "wb") as out: out.write(input_pdb_str)

if len(input_pdb_str) > 0:
  print()
  print(target_to_wrap)
  print()
  cmd = f"python sushimaki/sushimaki.py --barrel input.pdb"
  print(cmd)
  run_cmd(cmd)

wraps = []
for wrap in glob.glob('input_WRAP*.pdb'):
  wraps.append(wrap)

In [ ]:
#@title Select sushimaki WRAP for optimization
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display, clear_output

if len(wraps) == 0:
  raise Exception("sushimaki failed to generate wraps. Make sure your target has a transmembrane region predicted by DeepTMHMM")

current_wrap = ""
dropdown = widgets.Dropdown(
  options=wraps,
  description='wrap:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(wrap):
  global current_wrap
  current_wrap = wrap
  clear_output(wait=True)
  print()
  print(current_wrap)
  view = py3Dmol.view(width=500, height=400)
  with open(wrap, "r") as f:
    pdb_data = f.read()
  view.addModel(pdb_data, 'pdb')
  view.setStyle({'cartoon': {'colorscheme': 'chain'}})
  view.zoomTo()
  view.show()

widgets.interact(on_dropdown_change, wrap=dropdown);

In [ ]:

#@title Run **ppi_iterative_opt** partial diffusion -> mpnn -> af2 optimization on selected wrap.
#@markdown Runtime depends on partial_T, partial_diffusions, total_traj, and cycles (~1 to many hours).

#@markdown Backbone diversity increases with partial_T. Hint: A value of 30 may help for larger or more difficult targets.
%%time
sushimaki_wrap = current_wrap
partial_T = 20 #@param ["10", "15", "20", "25", "30"] {type:"raw"}
partial_diffusions = 2 #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"] {type:"raw"}
total_traj = 1 #@param ["1", "2", "3", "4", "5"] {type:"raw"}
cycles = 1 #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"] {type:"raw"}

os.environ["DGLBACKEND"] = "pytorch"
cmd = f"python sushimaki/ppi_iterative_opt/ppi_iterative_opt.py --partial_T {partial_T} --partial_diffusions {partial_diffusions} --cycles {cycles} --total_traj {total_traj} {sushimaki_wrap}"
print(cmd)
run_cmd(cmd)


python sushimaki/ppi_iterative_opt/ppi_iterative_opt.py --partial_T 20 --partial_diffusions 1 --cycles 1 --total_traj 2 input_WRAP_barrel_n17_S18_nres8_looplen3_termlen3_r17.04_cheight20_N_rot0.pdb
/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:68: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr/local/lib/python3.10/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[2026-10-28 21:41:48,529][inference.model_runners][INFO] - Reading checkpoint from /content/sushimaki/ppi_iterative

In [ ]:
#@title Plot AF2 plddt_binder vs pae_interaction of ppi_iterative_opt wraps.
import pandas as pd
import matplotlib.pyplot as plt
os.system("cat ppi_iterative_opt_output/*_af2.sc | head -n 1 > af2.sc")
os.system("cat ppi_iterative_opt_output/*_af2.sc | grep -v plddt_total | sort -n -k3 >> af2.sc")
df_af2 = pd.read_csv('af2.sc', sep=r'\s+')

def scatter_hist(x, y, ax, ax_histx, ax_histy, color, xlabel, ylabel):
    # no labels
    ax_histx.tick_params(axis="x", labelbottom=False)
    ax_histy.tick_params(axis="y", labelleft=False)

    # the scatter plot:
    ax.scatter(x, y)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax_histx.hist(x, bins=100)
    ax_histy.hist(y, orientation='horizontal', bins=100)

# Start with a square Figure.
fig = plt.figure(figsize=(8, 4))
gs = fig.add_gridspec(2, 2,  width_ratios=(4, 1), height_ratios=(1, 4),
                      left=0.1, right=0.9, bottom=0.1, top=0.9,
                      wspace=0.05, hspace=0.05)
# Create the Axes.
ax = fig.add_subplot(gs[1, 0])
ax_histx = fig.add_subplot(gs[0, 0], sharex=ax)
ax_histy = fig.add_subplot(gs[1, 1], sharey=ax)
# Draw the scatter plot.
scatter_hist(df_af2['pae_interaction'],df_af2['plddt_binder'], ax, ax_histx, ax_histy, 'black', 'pae interaction', 'plddt binder')


In [ ]:
#@title Download AF2 WRAP.
af2_wraps = { 'Select to download': ''}
for i,r in df_af2.iterrows():
  name = target_to_wrap + '_' + r['description'].split('/')[-1]+'.pdb'
  af2_wraps[f"{name} pae_i: {r['pae_interaction']} plddt_binder: {r['plddt_binder']}"] = r['description']+'.pdb'

dropdown = widgets.Dropdown(
  options=af2_wraps,
  description='af2 wrap:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(af2_wrap):
  if os.path.exists(af2_wrap):
    name = target_to_wrap +'_' + af2_wrap.split('/')[-1]
    os.system(f"cp {af2_wrap} {name}")
    files.download(name)

widgets.interact(on_dropdown_change, af2_wrap=dropdown);